# 01_data_contract_260513

광일 v2 master file 기준 row-level / subscription-event-level 데이터 계약 감사입니다.

금지 범위: modeling, SHAP, Optuna, causal claim, source CSV 수정, duration anomaly 제외 처리.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import math
import os

import pandas as pd
import numpy as np

PARK_ROOT = Path(r"C:\\Code\\ott-churn-prediction\\park.ingyeom").resolve()
SOURCE_PATH = PARK_ROOT / "data" / "(광일)Membership_v2_with_derived_features.csv"
NOTEBOOK_PATH = PARK_ROOT / "notebook" / "01_data_contract_260513" / "01_data_contract_260513.ipynb"
BASE_OUTPUT_DIR = PARK_ROOT / "reports" / "audits" / "01_data_contract_260513"

EXPECTED = {
    "row_count": 23343,
    "column_count": 91,
    "total_missing_count": 0,
    "unique_USER_KEY_count": 23134,
    "duplicated_USER_KEY_row_count": 209,
    "duration_lt_21_count": 238,
    "duration_eq_0_count": 90,
    "duration_21_30_count": 0,
}

def is_inside(child: Path, parent: Path) -> bool:
    try:
        child.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False

BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if any(BASE_OUTPUT_DIR.iterdir()):
    OUTPUT_DIR = BASE_OUTPUT_DIR / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
else:
    OUTPUT_DIR = BASE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

source_exists = SOURCE_PATH.exists()
source_stat_before = SOURCE_PATH.stat() if source_exists else None
written_files = []

print("PARK_ROOT:", PARK_ROOT)
print("SOURCE_PATH:", SOURCE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("source_file_exists:", source_exists)

assert is_inside(SOURCE_PATH, PARK_ROOT), "Source file is outside park.ingyeom"
assert is_inside(OUTPUT_DIR, PARK_ROOT), "Output folder is outside park.ingyeom"
assert is_inside(NOTEBOOK_PATH, PARK_ROOT), "Notebook is outside park.ingyeom"

PARK_ROOT: C:\Code\ott-churn-prediction\park.ingyeom
SOURCE_PATH: C:\Code\ott-churn-prediction\park.ingyeom\data\(광일)Membership_v2_with_derived_features.csv
OUTPUT_DIR: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513
source_file_exists: True


In [2]:
df = pd.read_csv(SOURCE_PATH)

def save_csv(frame: pd.DataFrame, filename: str):
    path = OUTPUT_DIR / filename
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output: {path}")
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    written_files.append(path)
    print("saved:", path)
    return path

def normalize_rate(count, denom):
    if denom == 0 or pd.isna(denom):
        return np.nan
    return count / denom

row_count = int(len(df))
column_count = int(df.shape[1])
total_missing_count = int(df.isna().sum().sum())
duplicated_full_row_count = int(df.duplicated().sum())

if "USER_KEY" in df.columns:
    unique_user_key_count = int(df["USER_KEY"].nunique(dropna=True))
    duplicated_user_key_row_count = int(row_count - unique_user_key_count)
    user_key_counts = df["USER_KEY"].value_counts(dropna=False)
    duplicated_keys = user_key_counts[user_key_counts > 1]
    duplicated_user_key_key_count = int(len(duplicated_keys))
    rows_belonging_to_duplicated_user_key_values = int(user_key_counts[user_key_counts > 1].sum())
else:
    unique_user_key_count = np.nan
    duplicated_user_key_row_count = np.nan
    duplicated_user_key_key_count = np.nan
    rows_belonging_to_duplicated_user_key_values = np.nan

summary = pd.DataFrame([
    {"metric": "source_file_exists", "value": bool(source_exists)},
    {"metric": "source_file_path", "value": str(SOURCE_PATH)},
    {"metric": "source_file_name", "value": SOURCE_PATH.name},
    {"metric": "source_file_size_bytes", "value": int(source_stat_before.st_size) if source_stat_before else np.nan},
    {"metric": "row_count", "value": row_count},
    {"metric": "column_count", "value": column_count},
    {"metric": "total_missing_count", "value": total_missing_count},
    {"metric": "duplicated_full_row_count", "value": duplicated_full_row_count},
    {"metric": "unique_USER_KEY_count", "value": unique_user_key_count},
    {"metric": "duplicated_USER_KEY_row_count", "value": duplicated_user_key_row_count},
    {"metric": "duplicated_USER_KEY_key_count", "value": duplicated_user_key_key_count},
    {"metric": "rows_belonging_to_duplicated_USER_KEY_values", "value": rows_belonging_to_duplicated_user_key_values},
])

column_inventory = pd.DataFrame({
    "column_name": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_rate": [float(df[c].isna().mean()) for c in df.columns],
    "nunique": [int(df[c].nunique(dropna=True)) for c in df.columns],
})

display(summary)
display(column_inventory.head(20))

save_csv(summary, "01_data_contract_summary.csv")
save_csv(column_inventory, "01_column_inventory.csv")

,metric,value
0,source_file_exists,True
1,source_file_path,C:\Code\ott-churn-prediction\park.ingyeom\data...
2,source_file_name,(광일)Membership_v2_with_derived_features.csv
3,source_file_size_bytes,10524468
4,row_count,23343
5,column_count,91
6,total_missing_count,0
7,duplicated_full_row_count,48
8,unique_USER_KEY_count,23134
9,duplicated_USER_KEY_row_count,209


,column_name,dtype,missing_count,missing_rate,nunique
0,USER_KEY,object,0,0.0,23134
1,product_code,object,0,0.0,46
2,price,float64,0,0.0,32
3,billing_method,int64,0,0.0,9
4,max_screen,float64,0,0.0,3
5,is_promotion,int64,0,0.0,2
6,is_churn_prevented,int64,0,0.0,2
7,payment_device,object,0,0.0,6
8,is_user_verified,int64,0,0.0,2
9,gender,object,0,0.0,3


saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_data_contract_summary.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_column_inventory.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/01_data_contract_260513/01_column_inventory.csv')

In [3]:
def distribution_table(column: str) -> pd.DataFrame:
    if column not in df.columns:
        return pd.DataFrame(columns=[column, "count", "rate", "note"])
    counts = df[column].value_counts(dropna=False).rename_axis(column).reset_index(name="count")
    counts["rate"] = counts["count"] / len(df)
    counts["note"] = "computed_from_source_csv"
    return counts

target_distribution = distribution_table("is_repurchase")
promotion_distribution = distribution_table("is_promotion")

if {"is_promotion", "is_repurchase"}.issubset(df.columns):
    promo_target_counts = df.groupby(["is_promotion", "is_repurchase"], dropna=False).size().reset_index(name="count")
    promo_target_counts["row_total_by_is_promotion"] = promo_target_counts.groupby("is_promotion", dropna=False)["count"].transform("sum")
    promo_target_counts["row_percentage_within_is_promotion"] = promo_target_counts["count"] / promo_target_counts["row_total_by_is_promotion"]
else:
    promo_target_counts = pd.DataFrame()


display(target_distribution)
display(promotion_distribution)
display(promo_target_counts)

save_csv(target_distribution, "01_target_distribution.csv")
save_csv(promotion_distribution, "01_promotion_distribution.csv")
save_csv(promo_target_counts, "01_promotion_target_2x2.csv")

,is_repurchase,count,rate,note
0,1,16702,0.715504,computed_from_source_csv
1,0,6641,0.284496,computed_from_source_csv


,is_promotion,count,rate,note
0,1,11955,0.512145,computed_from_source_csv
1,0,11388,0.487855,computed_from_source_csv


,is_promotion,is_repurchase,count,row_total_by_is_promotion,row_percentage_within_is_promotion
0,0,0,2746,11388,0.241131
1,0,1,8642,11388,0.758869
2,1,0,3895,11955,0.325805
3,1,1,8060,11955,0.674195


saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_target_distribution.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_promotion_distribution.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_promotion_target_2x2.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/01_data_contract_260513/01_promotion_target_2x2.csv')

In [4]:
date_parse_rows = []
parsed_dates = {}
for col in ["reg_date", "end_date"]:
    if col in df.columns:
        parsed = pd.to_datetime(df[col], errors="coerce")
        parsed_dates[col] = parsed
        non_missing = int(df[col].notna().sum())
        success = int(parsed.notna().sum())
        failure = int(non_missing - success)
        date_parse_rows.append({
            "column": col,
            "source_non_missing_count": non_missing,
            "parse_success_count": success,
            "parse_failure_count": failure,
            "parse_success_rate_among_non_missing": normalize_rate(success, non_missing),
        })
    else:
        date_parse_rows.append({
            "column": col,
            "source_non_missing_count": np.nan,
            "parse_success_count": np.nan,
            "parse_failure_count": np.nan,
            "parse_success_rate_among_non_missing": np.nan,
        })

date_parse_audit = pd.DataFrame(date_parse_rows)

if {"reg_date", "end_date"}.issubset(parsed_dates.keys()):
    duration_days = (parsed_dates["end_date"] - parsed_dates["reg_date"]).dt.days
else:
    duration_days = pd.Series([np.nan] * len(df), index=df.index, dtype="float")

duration_non_missing = duration_days.dropna()
duration_describe = duration_non_missing.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).reset_index()
duration_describe.columns = ["statistic", "value"]

duration_value_counts = duration_days.value_counts(dropna=False).rename_axis("duration_days").reset_index(name="count")
duration_value_counts["rate"] = duration_value_counts["count"] / len(df)

duration_lt_21 = duration_days < 21
duration_eq_0 = duration_days == 0
duration_21_30 = (duration_days >= 21) & (duration_days <= 30)
duration_gte_21 = duration_days >= 21

def bool_metric(name, mask):
    count = int(mask.fillna(False).sum())
    return {"metric": name, "count": count, "rate": normalize_rate(count, len(df))}

duration_anomaly_rows = [
    bool_metric("duration_lt_21", duration_lt_21),
    bool_metric("duration_eq_0", duration_eq_0),
    bool_metric("duration_21_30", duration_21_30),
    bool_metric("duration_gte_21", duration_gte_21),
]

def group_mask_table(group_cols, mask, metric_name):
    missing_cols = [c for c in group_cols if c not in df.columns]
    if missing_cols:
        return pd.DataFrame([{"grouping": "|".join(group_cols), "metric": metric_name, "note": f"missing columns: {missing_cols}"}])
    temp = df[group_cols].copy()
    temp[metric_name] = mask.fillna(False).astype(int)
    grouped = temp.groupby(group_cols, dropna=False)[metric_name].agg(["count", "sum"]).reset_index()
    grouped = grouped.rename(columns={"sum": f"{metric_name}_count", "count": "row_count"})
    grouped[f"{metric_name}_rate"] = grouped[f"{metric_name}_count"] / grouped["row_count"]
    grouped["grouping"] = "|".join(group_cols)
    return grouped

duration_anomaly_audit = pd.concat([
    pd.DataFrame(duration_anomaly_rows),
    group_mask_table(["is_promotion"], duration_lt_21, "duration_lt_21"),
    group_mask_table(["is_repurchase"], duration_lt_21, "duration_lt_21"),
    group_mask_table(["is_promotion", "is_repurchase"], duration_lt_21, "duration_lt_21"),
], ignore_index=True, sort=False)

display(date_parse_audit)
display(duration_describe)
display(duration_value_counts.head(30))
display(duration_anomaly_audit)

save_csv(date_parse_audit, "01_date_parse_audit.csv")
save_csv(duration_value_counts, "01_duration_distribution.csv")
save_csv(duration_anomaly_audit, "01_duration_anomaly_audit.csv")

,column,source_non_missing_count,parse_success_count,parse_failure_count,parse_success_rate_among_non_missing
0,reg_date,23343,23343,0,1.0
1,end_date,23343,23343,0,1.0


,statistic,value
0,count,23343.000000
1,mean,30.875894
2,std,2.908462
3,min,0.000000
4,1%,17.000000
5,5%,31.000000
6,25%,31.000000
7,50%,31.000000
8,75%,31.000000
9,95%,32.000000


,duration_days,count,rate
0,31,19280,0.825944
1,32,3825,0.163861
2,0,90,0.003856
3,1,51,0.002185
4,2,25,0.001071
5,3,16,0.000685
6,4,9,0.000386
7,8,7,0.000300
8,6,6,0.000257
9,9,6,0.000257


,metric,count,rate,is_promotion,row_count,duration_lt_21_count,duration_lt_21_rate,grouping,is_repurchase
0,duration_lt_21,238.0,0.010196,NaN,NaN,NaN,NaN,NaN,NaN
1,duration_eq_0,90.0,0.003856,NaN,NaN,NaN,NaN,NaN,NaN
2,duration_21_30,0.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
3,duration_gte_21,23105.0,0.989804,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,0.0,11388.0,187.0,0.016421,is_promotion,NaN
5,NaN,NaN,NaN,1.0,11955.0,51.0,0.004266,is_promotion,NaN
6,NaN,NaN,NaN,NaN,6641.0,117.0,0.017618,is_repurchase,0.0
7,NaN,NaN,NaN,NaN,16702.0,121.0,0.007245,is_repurchase,1.0
8,NaN,NaN,NaN,0.0,2746.0,89.0,0.032411,is_promotion|is_repurchase,0.0
9,NaN,NaN,NaN,0.0,8642.0,98.0,0.011340,is_promotion|is_repurchase,1.0


saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_date_parse_audit.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_duration_distribution.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_duration_anomaly_audit.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/01_data_contract_260513/01_duration_anomaly_audit.csv')

In [5]:
if "USER_KEY" in df.columns:
    dup_counts = df["USER_KEY"].value_counts(dropna=False)
    dup_distribution = dup_counts.value_counts().rename_axis("rows_per_USER_KEY").reset_index(name="USER_KEY_count")
    dup_distribution = dup_distribution.sort_values("rows_per_USER_KEY")
    duplicated_key_values = dup_counts[dup_counts > 1].index
    sample_dup_rows = df[df["USER_KEY"].isin(duplicated_key_values)].sort_values("USER_KEY").head(100).copy()
    user_key_duplicate_audit = pd.DataFrame([
        {"metric": "duplicated_USER_KEY_key_count", "value": int((dup_counts > 1).sum())},
        {"metric": "rows_belonging_to_duplicated_USER_KEY_values", "value": int(dup_counts[dup_counts > 1].sum())},
        {"metric": "duplicated_USER_KEY_row_count_extra_rows", "value": int(len(df) - df["USER_KEY"].nunique(dropna=True))},
        {"metric": "sample_duplicate_rows_saved", "value": int(len(sample_dup_rows))},
    ])
    user_key_duplicate_audit = pd.concat([
        user_key_duplicate_audit,
        dup_distribution.assign(metric="duplicate_count_distribution").rename(columns={"rows_per_USER_KEY": "value", "USER_KEY_count": "count"})[["metric", "value", "count"]]
    ], ignore_index=True, sort=False)
else:
    user_key_duplicate_audit = pd.DataFrame([{"metric": "USER_KEY_column_missing", "value": True}])
    sample_dup_rows = pd.DataFrame()

display(user_key_duplicate_audit)
display(sample_dup_rows.head())

save_csv(user_key_duplicate_audit, "01_user_key_duplicate_audit.csv")
save_csv(sample_dup_rows, "01_user_key_duplicate_samples.csv")

,metric,value,count
0,duplicated_USER_KEY_key_count,143,NaN
1,rows_belonging_to_duplicated_USER_KEY_values,352,NaN
2,duplicated_USER_KEY_row_count_extra_rows,209,NaN
3,sample_duplicate_rows_saved,100,NaN
4,duplicate_count_distribution,1,22991.0
5,duplicate_count_distribution,2,122.0
6,duplicate_count_distribution,3,9.0
7,duplicate_count_distribution,4,9.0
8,duplicate_count_distribution,10,1.0
9,duplicate_count_distribution,14,1.0


,USER_KEY,product_code,price,billing_method,max_screen,is_promotion,is_churn_prevented,payment_device,is_user_verified,gender,...,family_animation_ratio,drama_ratio,thriller_crime_ratio,sf_fantasy_ratio,comedy_ratio,romance_ratio,horror_ratio,documentary_ratio,historical_war_ratio,other_ratio
10666,01b73db46b0b3f6c24ca22dc4629ed41ed0d6ac4beccf8...,pk_1487,100.00,134,1.0,1,1,android,1,F,...,0.0,0.000000,0.0,0.22,0.0725,0.002500,0.0175,0.25,0.4375,0.0
11165,01b73db46b0b3f6c24ca22dc4629ed41ed0d6ac4beccf8...,pk_1488,10900.00,134,2.0,0,1,android,1,F,...,0.0,0.000000,0.0,0.22,0.0725,0.002500,0.0175,0.25,0.4375,0.0
5664,0258fd467c5765400d487d203f7c77afb597120daa9b1e...,pk_1508,9.99,140,1.0,0,0,ios,0,F,...,0.0,0.200000,0.0,0.00,0.6000,0.000000,0.0000,0.00,0.0000,0.0
5665,0258fd467c5765400d487d203f7c77afb597120daa9b1e...,pk_1508,9.99,140,1.0,0,0,ios,0,F,...,0.0,0.200000,0.0,0.00,0.6000,0.000000,0.0000,0.00,0.0000,0.0
9060,03207f956c0a2d363285286b1a9ede62e03373a9242253...,pk_2026,10900.00,151,2.0,0,0,android,0,N,...,0.0,0.592593,0.0,0.00,0.0000,0.407407,0.0000,0.00,0.0000,0.0


saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_user_key_duplicate_audit.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_user_key_duplicate_samples.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/01_data_contract_260513/01_user_key_duplicate_samples.csv')

In [6]:
actuals = {
    "row_count": row_count,
    "column_count": column_count,
    "total_missing_count": total_missing_count,
    "unique_USER_KEY_count": unique_user_key_count,
    "duplicated_USER_KEY_row_count": duplicated_user_key_row_count,
    "duration_lt_21_count": int(duration_lt_21.fillna(False).sum()),
    "duration_eq_0_count": int(duration_eq_0.fillna(False).sum()),
    "duration_21_30_count": int(duration_21_30.fillna(False).sum()),
}

expected_vs_actual = pd.DataFrame([
    {
        "check_name": key,
        "expected_value": expected,
        "actual_value": actuals.get(key, np.nan),
        "match": bool(actuals.get(key, np.nan) == expected),
        "note": "computed_from_source_csv_then_compared_to_docx_expected_values",
    }
    for key, expected in EXPECTED.items()
])

display(expected_vs_actual)
save_csv(expected_vs_actual, "01_expected_vs_actual_checks.csv")

,check_name,expected_value,actual_value,match,note
0,row_count,23343,23343,True,computed_from_source_csv_then_compared_to_docx...
1,column_count,91,91,True,computed_from_source_csv_then_compared_to_docx...
2,total_missing_count,0,0,True,computed_from_source_csv_then_compared_to_docx...
3,unique_USER_KEY_count,23134,23134,True,computed_from_source_csv_then_compared_to_docx...
4,duplicated_USER_KEY_row_count,209,209,True,computed_from_source_csv_then_compared_to_docx...
5,duration_lt_21_count,238,238,True,computed_from_source_csv_then_compared_to_docx...
6,duration_eq_0_count,90,90,True,computed_from_source_csv_then_compared_to_docx...
7,duration_21_30_count,0,0,True,computed_from_source_csv_then_compared_to_docx...


saved:

 C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_expected_vs_actual_checks.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/01_data_contract_260513/01_expected_vs_actual_checks.csv')

In [7]:
readme_text = f"""# 01_data_contract_260513

This is step 01 only.

- No modeling was performed.
- No SHAP was performed.
- No leakage/timing audit was performed yet.
- The source file is the 광일 v2 master file: `{SOURCE_PATH.name}`.
- The analysis unit should be treated as row-level / subscription-event-level because USER_KEY duplication exists.
- duration < 21 rows are only flagged here; no exclusion is applied in this goal.
- Next recommended step is 02_target_score_orientation_260513.

## Source

`{SOURCE_PATH}`

## Output Folder

`{OUTPUT_DIR}`
"""

readme_path = OUTPUT_DIR / "README.md"
if readme_path.exists():
    raise FileExistsError(f"Refusing to overwrite existing output: {readme_path}")
readme_path.write_text(readme_text, encoding="utf-8")
written_files.append(readme_path)
print("saved:", readme_path)

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\README.md


In [8]:
source_stat_after = SOURCE_PATH.stat() if SOURCE_PATH.exists() else None

required_outputs = [
    "01_data_contract_summary.csv",
    "01_column_inventory.csv",
    "01_target_distribution.csv",
    "01_promotion_distribution.csv",
    "01_promotion_target_2x2.csv",
    "01_date_parse_audit.csv",
    "01_duration_distribution.csv",
    "01_duration_anomaly_audit.csv",
    "01_user_key_duplicate_audit.csv",
    "01_user_key_duplicate_samples.csv",
    "01_expected_vs_actual_checks.csv",
    "README.md",
]

checks = []
def add_check(check, passed, evidence):
    checks.append({"check": check, "status": "PASS" if passed else "FAIL", "evidence": evidence})

add_check("source_file_exists", SOURCE_PATH.exists(), str(SOURCE_PATH))
add_check("source_file_inside_park_ingyeom", is_inside(SOURCE_PATH, PARK_ROOT), str(SOURCE_PATH))
add_check("output_folder_inside_park_ingyeom", is_inside(OUTPUT_DIR, PARK_ROOT), str(OUTPUT_DIR))
add_check("notebook_inside_park_ingyeom", is_inside(NOTEBOOK_PATH, PARK_ROOT), str(NOTEBOOK_PATH))
add_check("no_files_written_outside_park_ingyeom", all(is_inside(p, PARK_ROOT) for p in written_files), "all tracked written files are inside park.ingyeom")
add_check("no_py_script_created", not any(p.suffix.lower() == ".py" for p in written_files), "tracked written files contain no .py script")
add_check("no_existing_notebook_modified", True, "target notebook was newly created for this goal; no existing notebook was edited by notebook code")
add_check("no_source_csv_modified", source_stat_before and source_stat_after and source_stat_before.st_size == source_stat_after.st_size and source_stat_before.st_mtime == source_stat_after.st_mtime, "source file size and mtime unchanged")
add_check("no_modeling_performed", True, "no estimator, fit, predict, model training, or feature set creation performed")
add_check("no_shap_performed", True, "no shap import or SHAP computation performed")
add_check("row_count_recorded", "row_count" in summary["metric"].values, str(row_count))
add_check("column_count_recorded", "column_count" in summary["metric"].values, str(column_count))
add_check("total_missing_recorded", "total_missing_count" in summary["metric"].values, str(total_missing_count))
add_check("user_key_duplication_recorded", (OUTPUT_DIR / "01_user_key_duplicate_audit.csv").exists(), "01_user_key_duplicate_audit.csv")
add_check("target_distribution_recorded", (OUTPUT_DIR / "01_target_distribution.csv").exists(), "01_target_distribution.csv")
add_check("promotion_distribution_recorded", (OUTPUT_DIR / "01_promotion_distribution.csv").exists(), "01_promotion_distribution.csv")
add_check("promotion_target_2x2_recorded", (OUTPUT_DIR / "01_promotion_target_2x2.csv").exists(), "01_promotion_target_2x2.csv")
add_check("date_parse_audit_recorded", (OUTPUT_DIR / "01_date_parse_audit.csv").exists(), "01_date_parse_audit.csv")
add_check("duration_anomaly_recorded", (OUTPUT_DIR / "01_duration_anomaly_audit.csv").exists(), "01_duration_anomaly_audit.csv")
add_check("expected_vs_actual_checks_created", (OUTPUT_DIR / "01_expected_vs_actual_checks.csv").exists(), "01_expected_vs_actual_checks.csv")
add_check("readme_created", readme_path.exists(), "README.md")
for fname in required_outputs:
    add_check(f"required_output_exists::{fname}", (OUTPUT_DIR / fname).exists(), fname)

final_checks = pd.DataFrame(checks)
save_csv(final_checks, "01_final_checks.csv")

print("all_final_checks_passed:", bool((final_checks["status"] == "PASS").all()))
display(final_checks)

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513\01_final_checks.csv
all_final_checks_passed: True


,check,status,evidence
0,source_file_exists,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...
1,source_file_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...
2,output_folder_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
3,notebook_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\note...
4,no_files_written_outside_park_ingyeom,PASS,all tracked written files are inside park.ingyeom
5,no_py_script_created,PASS,tracked written files contain no .py script
6,no_existing_notebook_modified,PASS,target notebook was newly created for this goa...
7,no_source_csv_modified,PASS,source file size and mtime unchanged
8,no_modeling_performed,PASS,"no estimator, fit, predict, model training, or..."
9,no_shap_performed,PASS,no shap import or SHAP computation performed


## Final Summary

### Checked items

- Source file existence, path, name, and file size.
- Basic row-level / subscription-event-level data contract.
- Column inventory with dtype, missing count, missing rate, and nunique.
- `is_repurchase` target distribution.
- `is_promotion` distribution.
- `is_promotion` by `is_repurchase` 2x2 table and row percentage table.
- `reg_date` and `end_date` parse success/failure counts.
- Computed `duration_days = end_date - reg_date` distribution and anomaly flags.
- Duration < 21 rows by promotion, target, and promotion-target combination.
- USER_KEY duplication count, duplicate-count distribution, and duplicate-row sample up to 100 rows.
- Expected-vs-actual checks against the docx-provided expected values.
- Final execution checks.

### Unchecked items

- No modeling was performed.
- No SHAP was performed.
- No Optuna was performed.
- No leakage or timing audit was performed yet.
- No rows were excluded, including duration < 21 rows.
- No model-ready feature set was created.
- No causal effect was tested.

### Interpretation limits

- The analysis unit is row-level / subscription-event-level unless a later audit proves otherwise.
- Rows must not be called unique users because USER_KEY duplication exists.
- `is_repurchase=1` is treated as repurchase.
- Promotion comparisons in this step are descriptive distributions only, not causal evidence.
- Duration anomaly rows are flagged only; they are not removed in this step.

### Next recommended step

`02_target_score_orientation_260513`